> ⚠️ First, you need to **select the correct environment/kernel** to run the following notebook. Please select the kernel named **"Python (Main)"** ⚠️

# Data Preprocessing & 2D-CARE Training Dataset Generation

This notebook performs the data preprocessing steps required to obtain sub-pixel paired data, and then generates the training pairs needed for the 2D-CARE training part.

The standard protocol for preprocessing is as follows:

1. Splitting the stack acquired with the beam splitter into two,
2. Sub-pixel alignment using a coefficient file **previously calculated with PALMTracer**,
3. **In the case of 2D-CARE training**, preparing the dataset for localization-informed training.

## Prerequisites if running for the first time

If you are **running the notebook for the first time**, you will first need to install the packages required to smoothly run the whole pipeline. 

> If you don't have Python installed on the machine, please install it beforehand. For this repository, you will need to install a version of **Python 3.9**. We used 3.9.25 version.


### Windows
In the main folder, there is an *install.bat* file (Windows). Double-clicking it is enough to install the necessary virtual environments and libraries.

### MacOS
Execute the following line one by one in a Terminal, while being in root directory:

```sh
chmod +x install.sh
./install.sh
```

> ⚠️ If you have a **libomp** error, you will need to install it with **brew**. ⚠️

Once done, **three** virtual environments should be visible: **venv**, **venv_aydin** and **venv_csbdeep**.

## Importing libraries
In the following section, we will load Python librairies required. Run the next cell to load librairies and dependencies:

In [13]:
# Run to load libraries

import os
import sys
import tifffile

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from ipyfilechooser import FileChooser
from IPython.display import display, clear_output
from ipywidgets import widgets

sys.path.append(os.path.abspath('..'))

from utils.split_raw_image import split_raw_image


## Splitting dataset in two
An optical system containing a beam splitter is used to simultaneously generate high and low SNR images. These images are on the same sensor, so **it is necessary to split them into two**. This can be done manually (using ImageJ, for example), taking care to note the ROI size and the (0,0) coordinates of the two images relative to the original image.

The following code can load the path of one image and split it in two using:

1. Initial XY coordinates for the ROI,
2. Size of the ROI, 
3. Gap between the two images.

First, load an image:

In [ ]:
# Load the image

IMAGE_PATH = None

fc = FileChooser(os.getcwd())
fc.show_only_dirs = False
fc.filter_pattern = ['*.tif', '*.tiff', '*.stk']
fc.title = '<b>Choose raw image (.tif, .stk) :</b>'

def check_file(chooser):
    global IMAGE_PATH
    clear_output(wait=True)
    display(fc)
    
    selected_file = chooser.selected
    
    if selected_file is None:
        return
        
    if os.path.isfile(selected_file):
        IMAGE_PATH = selected_file
        print(f"Image loaded : {IMAGE_PATH}")
        print("Can now perform image splitting.")
    else:
        IMAGE_PATH = None
        print("Error : Please select an image, not a repertory or something else.")

fc.register_callback(check_file)
display(fc)

FileChooser(path='/Users/aneuhaus/Desktop/SCOL/SCOL/notebooks', filename='', title='<b>Choose raw image (.tif,…

You can know perform the splitting of the raw image loaded. Please explore your data with ImageJ first to enter proper values for ROI size and initial XY coordinates.

In [ ]:
# Perform splitting
if IMAGE_PATH is None:
    print("WARNING : Please select an image first above.")
else:
    img_temp = tifffile.imread(IMAGE_PATH)
    img_display = img_temp[0] if img_temp.ndim > 2 else img_temp
    img_height_max, img_width_max = img_display.shape
    default_roi_h = (img_height_max // 2) - 15 

    # widgets
    w_X = widgets.IntText(value=0, description="Initial X:")
    w_Y = widgets.IntText(value=0, description="Initial Y:")
    w_W = widgets.IntText(value=img_width_max, description="Width :")
    w_H = widgets.IntText(value=default_roi_h, description="Height :")
    w_GAP = widgets.IntText(value=30, description="Gap:")
    btn_validate = widgets.Button(description="Click to split", button_style='success', icon='check')

    def update_preview(b=None):
        X, Y, W, H, GAP = w_X.value, w_Y.value, w_W.value, w_H.value, w_GAP.value
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        ax.imshow(img_display, cmap="gray")
        rect_high = patches.Rectangle((X, Y), W, H, linewidth=2, edgecolor='red', facecolor='none', label='High SNR')
        ax.add_patch(rect_high)
        y2_start = Y + H + GAP
        rect_low = patches.Rectangle((X, y2_start), W, H, linewidth=2, edgecolor='dodgerblue', facecolor='none', label='Low SNR')
        ax.add_patch(rect_low)
        ax.legend(loc='upper right')
        ax.set_title(f"Aperçu : {img_width_max}x{img_height_max} px")
        plt.show()

        if y2_start + H > img_height_max or X + W > img_width_max:
            print("Error : ROI size are to small/large.")
        else:
            print(f"Ready to save.")

    def on_button_clicked(b):
        update_preview()
        print("Saving...")

    btn_validate.on_click(on_button_clicked)

    ui = widgets.VBox([
        widgets.HBox([w_X, w_Y, w_W]),
        widgets.HBox([w_H, w_GAP]),
        btn_validate
    ])
    
    display(ui)
    update_preview()